# Intent Classifier Demo

This notebook demonstrates how to use the intent classifier for SQL and chart requests in a chatbot application.

## Setup

First, let's import the necessary modules and set up our environment.

In [1]:
# !pip install --upgrade langchain_aws langchain_experimental langchain_community langchain_core

In [1]:
import sys
import os
import json

# Add the src directory to the Python path so we can import our modules
sys.path.append(os.path.abspath('../'))

# Import our intent classifier modules
from src.intent_classifier.classifier import IntentClassifier
from src.intent_classifier.schema import (
    FirstQueryIntentType, 
    FollowupQueryIntentType,
    SalutationType
)
from src.intent_classifier.utils import format_context, context_to_string

## Setup AWS Credentials for Bedrock

For using Amazon Bedrock, you need to set up your AWS credentials. You can do this in several ways:

1. Using environment variables
2. Using AWS configuration files
3. Using AWS IAM roles

For this notebook, we'll use environment variables. Make sure to replace the placeholders with your actual AWS credentials.

In [2]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv(override=True)

print("AWS credentials loaded successfully")

AWS credentials loaded successfully


In [3]:
os.getenv("ANTHROPIC_MODEL_ID")

'us.anthropic.claude-3-5-sonnet-20241022-v2:0'

## Initialize the Intent Classifier

Now, let's initialize our intent classifier with Amazon Bedrock.

In [4]:
# Set AWS region
os.environ["AWS_DEFAULT_REGION"] = os.getenv("AWS_REGION_NAME")
# Initialize the classifier with the default Bedrock model
classifier = IntentClassifier(model_id="us.anthropic.claude-3-5-sonnet-20241022-v2:0", temperature=0.1)

# If you want to use a different model, you can specify it
# classifier = IntentClassifier(model_id="anthropic.claude-3-haiku-20240307-v1:0", temperature=0.1)

## Test First Query Intent Classification

Let's test the classifier with some example first queries.

In [5]:
# Example first queries
first_queries = [
    "Show me the total sales by region",
    "Get me a breakdown of revenue by product category and create a bar chart",
    "Write a SQL query to find all customers who haven't ordered in the last 6 months",
    "Create a query to analyze monthly sales trends and visualize it as a line chart"
]

# Classify each query
first_query_results = []

for query in first_queries:
    print(f"\nQuery: {query}")
    intent = classifier.classify_first_query(query)
    print(f"Intent Type: {intent.intent_type}")
    first_query_results.append(intent)


Query: Show me the total sales by region
Intent Type: FirstQueryIntentType.SQL_ONLY

Query: Get me a breakdown of revenue by product category and create a bar chart
Intent Type: FirstQueryIntentType.SQL_ONLY

Query: Get me a breakdown of revenue by product category and create a bar chart
Intent Type: FirstQueryIntentType.SQL_AND_CHART

Query: Write a SQL query to find all customers who haven't ordered in the last 6 months
Intent Type: FirstQueryIntentType.SQL_AND_CHART

Query: Write a SQL query to find all customers who haven't ordered in the last 6 months
Intent Type: FirstQueryIntentType.SQL_ONLY

Query: Create a query to analyze monthly sales trends and visualize it as a line chart
Intent Type: FirstQueryIntentType.SQL_ONLY

Query: Create a query to analyze monthly sales trends and visualize it as a line chart
Intent Type: FirstQueryIntentType.SQL_AND_CHART
Intent Type: FirstQueryIntentType.SQL_AND_CHART


## Test Follow-up Query Intent Classification

Now, let's simulate some follow-up queries with context from previous interactions.

In [6]:
# Create some example context from a previous interaction
previous_sql = """
SELECT region, SUM(sales) as total_sales
FROM sales_data
GROUP BY region
ORDER BY total_sales DESC
"""

previous_chart_config = {
    "type": "bar",
    "x_axis": "region",
    "y_axis": "total_sales",
    "title": "Total Sales by Region",
    "color": "blue"
}

# Format the context
context = format_context(
    previous_sql=previous_sql,
    previous_chart_config=previous_chart_config,
    previous_intents=[{"raw_query": first_queries[1], "intent_type": "sql_and_chart"}]
)

# Pretty print the context
print("Example Context:")
print(context_to_string(context))

Example Context:
Previous SQL Query:

SELECT region, SUM(sales) as total_sales
FROM sales_data
GROUP BY region
ORDER BY total_sales DESC


Previous Chart Configuration:
{
  "type": "bar",
  "x_axis": "region",
  "y_axis": "total_sales",
  "title": "Total Sales by Region",
  "color": "blue"
}

Previous Intents:
- 1. Get me a breakdown of revenue by product category and create a bar chart (Intent: sql_and_chart)


In [7]:
# Example follow-up queries
followup_queries = [
    "Add a filter for the year 2024",
    "Change the chart type to pie chart",
    "Include product category in the query and update the chart to show a stacked bar chart"
]

# Classify each follow-up query
followup_query_results = []

for query in followup_queries:
    print(f"\nQuery: {query}")
    intent = classifier.classify_followup_query(query, context)
    print(f"Intent Type: {intent.intent_type}")
    followup_query_results.append(intent)


Query: Add a filter for the year 2024
Intent Type: FollowupQueryIntentType.MODIFY_SQL_AND_CHART

Query: Change the chart type to pie chart
Intent Type: FollowupQueryIntentType.MODIFY_SQL_AND_CHART

Query: Change the chart type to pie chart
Intent Type: FollowupQueryIntentType.MODIFY_CHART_ONLY

Query: Include product category in the query and update the chart to show a stacked bar chart
Intent Type: FollowupQueryIntentType.MODIFY_CHART_ONLY

Query: Include product category in the query and update the chart to show a stacked bar chart
Intent Type: FollowupQueryIntentType.MODIFY_SQL_AND_CHART
Intent Type: FollowupQueryIntentType.MODIFY_SQL_AND_CHART


## Complete Conversation Flow Example

Here's an example of how to use the intent classifier in a complete conversation flow.

In [15]:
def simulate_conversation():
    # Initialize the classifier
    classifier = IntentClassifier()
    
    # Conversation state
    conversation_context = {}
    previous_intents = []
    current_sql = None
    current_chart_config = None
    is_first_query = True
    
    # Simulated user queries
    queries = [
        "Show me total sales by product category as a bar chart",  # First query
        "Add a filter for transactions after January 2024",        # Modify SQL only
        "Change the chart to a pie chart",                         # Modify chart only
        "Group by region instead and make it a stacked bar chart"  # Modify both
    ]
    
    for i, query in enumerate(queries):
        print(f"\n===== Query {i+1} =====")
        print(f"User: {query}")
        
        # Classify the intent
        if is_first_query:
            intent = classifier.classify_first_query(query)
            print(f"Detected Intent: {intent.intent_type}")
            is_first_query = False
            
            # Simulate processing the first query
            if intent.intent_type == FirstQueryIntentType.SQL_ONLY:
                print("System: Generating SQL query...")
                current_sql = """
                SELECT product_category, SUM(sales) as total_sales
                FROM sales_data
                GROUP BY product_category
                ORDER BY total_sales DESC
                """
                print(f"Generated SQL:\n{current_sql}")
            else:  # SQL_AND_CHART
                print("System: Generating SQL query and chart...")
                current_sql = """
                SELECT product_category, SUM(sales) as total_sales
                FROM sales_data
                GROUP BY product_category
                ORDER BY total_sales DESC
                """
                current_chart_config = {
                    "type": "bar",
                    "x_axis": "product_category",
                    "y_axis": "total_sales",
                    "title": "Total Sales by Product Category"
                }
                print(f"Generated SQL:\n{current_sql}")
                print(f"Generated Chart Config:\n{json.dumps(current_chart_config, indent=2)}")
        else:
            # Update context with previous state
            conversation_context = format_context(
                previous_sql=current_sql,
                previous_chart_config=current_chart_config,
                previous_intents=previous_intents
            )
            
            intent = classifier.classify_followup_query(query, conversation_context)
            print(f"Detected Intent: {intent.intent_type}")
                
            if intent.intent_type == FollowupQueryIntentType.MODIFY_CHART_ONLY:
                print("System: Modifying chart configuration...")
                # Simulate chart modification
                if "pie chart" in query:
                    current_chart_config["type"] = "pie"
                print(f"Modified Chart Config:\n{json.dumps(current_chart_config, indent=2)}")
                
            else:  # MODIFY_SQL_AND_CHART
                print("System: Modifying both SQL query and chart...")
                # Simulate both SQL and chart modification
                if "region" in query:
                    current_sql = """
                    SELECT region, product_category, SUM(sales) as total_sales
                    FROM sales_data
                    WHERE transaction_date >= '2024-01-01'
                    GROUP BY region, product_category
                    ORDER BY region, total_sales DESC
                    """
                    current_chart_config = {
                        "type": "stacked_bar",
                        "x_axis": "region",
                        "y_axis": "total_sales",
                        "group_by": "product_category",
                        "title": "Total Sales by Region and Product Category"
                    }
                print(f"Modified SQL:\n{current_sql}")
                print(f"Modified Chart Config:\n{json.dumps(current_chart_config, indent=2)}")
        
        # Store the intent for context in the next iteration
        previous_intents.append({
            "raw_query": query,
            "intent_type": intent.intent_type
        })
    
    print("\n===== Conversation Completed =====")

# Run the simulated conversation
simulate_conversation()


===== Query 1 =====
User: Show me total sales by product category as a bar chart
Detected Intent: FirstQueryIntentType.SQL_AND_CHART
System: Generating SQL query and chart...
Generated SQL:

                SELECT product_category, SUM(sales) as total_sales
                FROM sales_data
                GROUP BY product_category
                ORDER BY total_sales DESC
                
Generated Chart Config:
{
  "type": "bar",
  "x_axis": "product_category",
  "y_axis": "total_sales",
  "title": "Total Sales by Product Category"
}

===== Query 2 =====
User: Add a filter for transactions after January 2024
Detected Intent: FollowupQueryIntentType.MODIFY_SQL_AND_CHART
System: Modifying both SQL query and chart...
Modified SQL:

                SELECT product_category, SUM(sales) as total_sales
                FROM sales_data
                GROUP BY product_category
                ORDER BY total_sales DESC
                
Modified Chart Config:
{
  "type": "bar",
  "x_axis": "produ

## Handling Real-world Chatbot Integration

In a real-world application, you'd integrate this intent classifier with:

1. An LLM-based SQL generator
2. A chart configuration generator
3. A database connector to execute the SQL
4. A visualization library to render the charts

Here's a basic pseudo-code for how that integration might look:

In [9]:
def process_user_query(query, conversation_state):
    """Process a user query in a chatbot context.
    
    Args:
        query: The user's query text
        conversation_state: Dictionary tracking the state of the conversation
        
    Returns:
        Updated conversation state and response to the user
    """
    # Extract current state
    is_first_query = conversation_state.get("is_first_query", True)
    current_sql = conversation_state.get("current_sql")
    current_chart_config = conversation_state.get("current_chart_config")
    previous_intents = conversation_state.get("previous_intents", [])
    
    # Initialize the intent classifier
    classifier = IntentClassifier()
    
    # Classify the query intent
    if is_first_query:
        intent = classifier.classify_first_query(query)
        
        # Handle first query intent
        if intent.intent_type == FirstQueryIntentType.SQL_ONLY:
            # Generate SQL only
            current_sql = generate_sql(query)  # Placeholder for SQL generation function
            sql_results = execute_sql(current_sql)  # Placeholder for SQL execution
            response = format_sql_results(sql_results)  # Placeholder for formatting
            
        else:  # SQL_AND_CHART
            # Generate SQL and chart
            current_sql = generate_sql(query)  # Placeholder for SQL generation
            sql_results = execute_sql(current_sql)  # Placeholder for SQL execution
            current_chart_config = generate_chart_config(query, sql_results)  # Placeholder
            chart_image = render_chart(sql_results, current_chart_config)  # Placeholder
            response = format_sql_and_chart_results(sql_results, chart_image)  # Placeholder
            
    else:
        # Format context from previous state
        context = format_context(
            previous_sql=current_sql,
            previous_chart_config=current_chart_config,
            previous_intents=previous_intents
        )
        
        intent = classifier.classify_followup_query(query, context)
        
        # Handle follow-up query intent
        if intent.intent_type == FollowupQueryIntentType.MODIFY_SQL_ONLY:
            # Modify SQL only
            current_sql = modify_sql(current_sql, query)  # Placeholder
            sql_results = execute_sql(current_sql)  # Placeholder
            response = format_sql_results(sql_results)  # Placeholder
            
        elif intent.intent_type == FollowupQueryIntentType.MODIFY_CHART_ONLY:
            # Modify chart only - reuse existing SQL results
            sql_results = execute_sql(current_sql)  # Placeholder
            current_chart_config = modify_chart_config(current_chart_config, query)  # Placeholder
            chart_image = render_chart(sql_results, current_chart_config)  # Placeholder
            response = format_chart_results(chart_image)  # Placeholder
            
        else:  # MODIFY_BOTH
            # Modify both SQL and chart
            current_sql = modify_sql(current_sql, query)  # Placeholder
            sql_results = execute_sql(current_sql)  # Placeholder
            current_chart_config = modify_chart_config(current_chart_config, query)  # Placeholder
            chart_image = render_chart(sql_results, current_chart_config)  # Placeholder
            response = format_sql_and_chart_results(sql_results, chart_image)  # Placeholder
    
    # Store the intent for context in future queries
    previous_intents.append({
        "raw_query": query,
        "intent_type": intent.intent_type
    })
    
    # Update conversation state
    conversation_state.update({
        "is_first_query": False,
        "current_sql": current_sql,
        "current_chart_config": current_chart_config,
        "previous_intents": previous_intents
    })
    
    return conversation_state, response

# Placeholder functions for the real implementation
def generate_sql(query):
    """Generate SQL from a natural language query."""
    return "SELECT * FROM table"  # This would be implemented with an LLM

def execute_sql(sql):
    """Execute SQL and return results."""
    return [{}]  # This would connect to a real database

def generate_chart_config(query, data):
    """Generate chart configuration from query and data."""
    return {"type": "bar"}  # This would be implemented with an LLM

def modify_sql(current_sql, query):
    """Modify existing SQL based on query."""
    return current_sql  # This would be implemented with an LLM

def modify_chart_config(current_config, query):
    """Modify existing chart config based on query."""
    return current_config  # This would be implemented with an LLM

def render_chart(data, config):
    """Render a chart from data and config."""
    return "chart_image.png"  # This would use a visualization library

def format_sql_results(results):
    """Format SQL results for display."""
    return "Here are your results..."  # This would format results nicely

def format_chart_results(chart):
    """Format chart for display."""
    return "Here is your chart..."  # This would include the chart image

def format_sql_and_chart_results(results, chart):
    """Format SQL results and chart for display."""
    return "Here are your results and chart..."  # This would format both

## Test Salutation Intent Classification

Let's test the classifier to identify different types of salutations.

In [16]:
# Example salutation messages
salutation_messages = [
    "Hi there!",                      # Greeting
    "Hello, how are you today?",      # Greeting
    "Thanks for your help!",          # Thanks
    "Thank you for the assistance",   # Thanks
    "Goodbye, talk to you later",     # Goodbye
    "See you tomorrow",               # Goodbye
    "I appreciate your work",         # Thanks/Other
    "Show me sales data for 2024"     # Not a salutation (actual query)
]

# Classify each message
for message in salutation_messages:
    print(f"\nMessage: {message}")
    salutation_intent = classifier.classify_salutation(message)
    
    if salutation_intent:
        print(f"Detected as salutation - Type: {salutation_intent.intent_type}")
    else:
        print("Not detected as a salutation - This is likely a query")


Message: Hi there!
Detected as salutation - Type: SalutationType.GREETING

Message: Hello, how are you today?
Detected as salutation - Type: SalutationType.GREETING

Message: Thanks for your help!
Detected as salutation - Type: SalutationType.THANKS

Message: Thank you for the assistance
Detected as salutation - Type: SalutationType.THANKS

Message: Goodbye, talk to you later
Detected as salutation - Type: SalutationType.GOODBYE

Message: See you tomorrow
Detected as salutation - Type: SalutationType.GOODBYE

Message: I appreciate your work
Detected as salutation - Type: SalutationType.THANKS

Message: Show me sales data for 2024
Not detected as a salutation - This is likely a query


## Complete Conversation Flow with Salutation Detection

Here's an enhanced example showing how to use the salutation detection in a complete conversation flow.

In [17]:
import json

def simulate_conversation_with_salutations():
    global classifier
    
    conversation_context = {}
    previous_intents = []
    current_sql = None
    current_chart_config = None
    is_first_query = True
    
    messages = [
        "Hello there!",                                        
        "Show me total sales by product category as a bar chart",  
        "Thanks for that!",                                    
        "Add a filter for transactions after January 2024",        
        "Change the chart to a pie chart please",                  
        "Thank you, that's perfect!",                           
        "Goodbye, that's all I needed"                          
    ]
    
    for i, message in enumerate(messages):
        print(f"\n===== Message {i+1} =====")
        print(f"User: {message}")
        
        try:
            salutation_intent = classifier.classify_salutation(message)
            
            if salutation_intent and salutation_intent.intent_type:
                print(f"Detected salutation intent: {salutation_intent.intent_type}")
                
                if salutation_intent.intent_type == SalutationType.GREETING:
                    print("System: Hello! How can I help you with SQL queries or visualizations today?")
                elif salutation_intent.intent_type == SalutationType.THANKS:
                    print("System: You're welcome! Is there anything else you'd like to know or modify?")
                elif salutation_intent.intent_type == SalutationType.GOODBYE:
                    print("System: Goodbye! Feel free to come back if you need more assistance.")
                else:
                    print("System: I'm here to help with your SQL and visualization needs.")
                    
                previous_intents.append({
                    "raw_query": message,
                    "intent_type": str(salutation_intent.intent_type)
                })
                continue
            
            if is_first_query:
                intent = classifier.classify_first_query(message)
                if intent:
                    print(f"Detected first query intent: {intent.intent_type}")
                    
                is_first_query = False
                
                if intent and intent.intent_type == FirstQueryIntentType.SQL_AND_CHART:
                    print("System: Generating SQL query and chart...")
                    current_sql = """
                    SELECT product_category, SUM(sales) as total_sales
                    FROM sales_data
                    GROUP BY product_category
                    ORDER BY total_sales DESC
                    """
                    current_chart_config = {
                        "type": "bar",
                        "x_axis": "product_category",
                        "y_axis": "total_sales",
                        "title": "Total Sales by Product Category"
                    }
                    print(f"Generated SQL:\n{current_sql}")
                    print(f"Generated Chart Config:\n{json.dumps(current_chart_config, indent=2)}")
            else:
                conversation_context = {
                    "previous_sql": current_sql,
                    "previous_chart_config": current_chart_config,
                    "previous_intents": previous_intents
                }
                
                intent = classifier.classify_followup_query(message, conversation_context)
                if intent:
                    print(f"Detected followup intent: {intent.intent_type}")
                    
                if intent and intent.intent_type == FollowupQueryIntentType.MODIFY_CHART_ONLY:
                    if "pie chart" in message.lower():
                        current_chart_config["type"] = "pie"
                        print("System: Modified chart to pie chart")
                        print(f"Modified Chart Config:\n{json.dumps(current_chart_config, indent=2)}")
                elif intent and intent.intent_type == FollowupQueryIntentType.MODIFY_SQL_AND_CHART:
                    if "January 2024" in message:
                        current_sql = current_sql.replace(
                            "FROM sales_data",
                            "FROM sales_data\nWHERE transaction_date >= '2024-01-01'"
                        )
                        print("System: Added date filter")
                        print(f"Modified SQL:\n{current_sql}")
                        print("System: Modified Chart Config")
                        print(f"Modified Chart Config:\n{json.dumps(current_chart_config, indent=2)}")
            
            if intent:
                previous_intents.append({
                    "raw_query": message,
                    "intent_type": str(intent.intent_type)
                })
                
        except Exception as e:
            print(f"System: Error processing message: {str(e)}")
    
    print("\n===== Conversation Completed =====")

simulate_conversation_with_salutations()



===== Message 1 =====
User: Hello there!
Detected salutation intent: SalutationType.GREETING
System: Hello! How can I help you with SQL queries or visualizations today?

===== Message 2 =====
User: Show me total sales by product category as a bar chart
Detected first query intent: FirstQueryIntentType.SQL_AND_CHART
System: Generating SQL query and chart...
Generated SQL:

                    SELECT product_category, SUM(sales) as total_sales
                    FROM sales_data
                    GROUP BY product_category
                    ORDER BY total_sales DESC
                    
Generated Chart Config:
{
  "type": "bar",
  "x_axis": "product_category",
  "y_axis": "total_sales",
  "title": "Total Sales by Product Category"
}

===== Message 3 =====
User: Thanks for that!
Detected salutation intent: SalutationType.THANKS
System: You're welcome! Is there anything else you'd like to know or modify?

===== Message 4 =====
User: Add a filter for transactions after January 2024
Dete

## Enhanced Chatbot Integration with Salutation Detection

Here's an updated pseudo-code example for a real-world chatbot that includes salutation detection:

In [ ]:
def enhanced_process_user_message(message, conversation_state):
    """Process a user message in a chatbot context with salutation detection.
    
    Args:
        message: The user's message text
        conversation_state: Dictionary tracking the state of the conversation
        
    Returns:
        Updated conversation state and response to the user
    """
    # Extract current state
    is_first_query = conversation_state.get("is_first_query", True)
    current_sql = conversation_state.get("current_sql")
    current_chart_config = conversation_state.get("current_chart_config")
    previous_intents = conversation_state.get("previous_intents", [])
    
    # Initialize the intent classifier
    classifier = IntentClassifier()
    
    # First check if it's a salutation
    salutation_intent = classifier.classify_salutation(message)
    
    if salutation_intent:
        # Handle different types of salutations
        if salutation_intent.intent_type == SalutationType.GREETING:
            response = "Hello! How can I help you with SQL queries or visualizations today?"
        elif salutation_intent.intent_type == SalutationType.THANKS:
            response = "You're welcome! Is there anything else you'd like to know or modify?"
        elif salutation_intent.intent_type == SalutationType.GOODBYE:
            response = "Goodbye! Feel free to come back if you need more assistance."
        else:  # OTHER
            response = "I'm here to help with your SQL and visualization needs."
        
        # Store the intent and update conversation state
        previous_intents.append({
            "raw_query": message,
            "intent_type": str(salutation_intent.intent_type)
        })
        
        conversation_state["previous_intents"] = previous_intents
        return conversation_state, response
    
    # If not a salutation, process as a normal query (using the process_user_query function)
    return process_user_query(message, conversation_state)

# Note: This function would use the process_user_query function defined earlier
# to handle non-salutation messages

## Conclusion

This notebook has demonstrated how to use the intent classifier to identify user intentions in SQL and chart-related conversations. The key features demonstrated include:

1. Distinguishing between first queries and follow-up queries
2. Classifying queries into specific intent types
3. Maintaining conversation context between interactions
4. Using the intent classifications to drive appropriate responses

In a production environment, you would integrate this with actual SQL generation, chart configuration, database connectivity, and visualization components.